SDG 1: No Poverty.

1.Data Loading and Initial Exploratory Analysis

In [3]:
from google.colab import drive
import pandas as pd
drive.mount('/content/drive')
file_path = '/content/drive/MyDrive/Colab Notebooks/train.csv/train.csv'

train = pd.read_csv(file_path)
print("資料大小:", train.shape)
train.head()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
資料大小: (9557, 143)


,Id,v2a1,hacdor,rooms,hacapo,v14a,refrig,v18q,v18q1,r4h1,...,SQBescolari,SQBage,SQBhogar_total,SQBedjefe,SQBhogar_nin,SQBovercrowding,SQBdependency,SQBmeaned,agesq,Target
0,ID_279628684,190000.0,0,3,0,1,1,0,NaN,0,...,100,1849,1,100,0,1.000000,0.0,100.0,1849,4
1,ID_f29eb3ddd,135000.0,0,4,0,1,1,1,1.0,0,...,144,4489,1,144,0,1.000000,64.0,144.0,4489,4
2,ID_68de51c94,NaN,0,8,0,1,1,0,NaN,0,...,121,8464,1,0,0,0.250000,64.0,121.0,8464,4
3,ID_d671db89c,180000.0,0,5,0,1,1,1,1.0,0,...,81,289,16,121,4,1.777778,1.0,121.0,289,4
4,ID_d56d6f5f5,180000.0,0,5,0,1,1,1,1.0,0,...,121,1369,16,121,4,1.777778,1.0,121.0,1369,4


2. Data Cleaning and Feature Preprocessing

In [4]:
import numpy as np

mapping = {"yes": 1, "no": 0}

cols_to_fix = ['dependency', 'edjefe', 'edjefa']
for col in cols_to_fix:
    train[col] = train[col].replace(mapping).astype(float)

print("✅ Categorical mapping complete: 'yes/no' converted to 1/0.")


train['v18q1'] = train['v18q1'].fillna(0)


train['v2a1'] = train['v2a1'].fillna(0)


train['rez_esc'] = train['rez_esc'].fillna(0)


train['meaneduc'] = train['meaneduc'].fillna(train['meaneduc'].median())
train['SQBmeaned'] = train['SQBmeaned'].fillna(train['SQBmeaned'].median())

print("✅ Missing value imputation complete.")


print("\nRemaining Missing Values Check:")
print(train[['v18q1', 'v2a1', 'rez_esc', 'meaneduc']].isnull().sum())

✅ Categorical mapping complete: 'yes/no' converted to 1/0.
✅ Missing value imputation complete.

Remaining Missing Values Check:
v18q1       0
v2a1        0
rez_esc     0
meaneduc    0
dtype: int64


3.Household-Level Data Aggregation

In [7]:
# Define aggregation rules based on feature characteristics
aggregation_rules = {
    'meaneduc': 'mean',   # Average education level of the household
    'hacdor': 'max',      # Overcrowding indicator
    'v18q1': 'max',       # Number of tablets
    'refrig': 'max',      # Presence of a refrigerator
    'computer': 'max',    # Presence of a computer
    'television': 'max',  # Presence of a television
    'mobilephone': 'max', # Presence of a mobile phone
    'qmobilephone': 'max',# Total count of mobile phones
    'age': ['mean', 'min', 'max'] # Age demographics (Avg, Min, Max)
}

# Execute aggregation grouped by the household ID (idhogar)
df_household = train.groupby('idhogar').agg(aggregation_rules)

# Flatten the multi-level columns created by aggregation
df_household.columns = ['_'.join(col).strip() for col in df_household.columns.values]

# Join the Target variable back to the aggregated data
target = train.groupby('idhogar')['Target'].first()
df_final = df_household.join(target)

print(f"Aggregation complete. Individual rows: {len(train)} -> Household rows: {len(df_final)}")

Aggregation complete. Individual rows: 9557 -> Household rows: 2988


4.Model Training and Hyperparameter Tuning

In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import classification_report, confusion_matrix

# Prepare feature matrix (X) and target vector (y)
X = df_final.drop('Target', axis=1)
y = df_final['Target']

# Splitting data into Training and Testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

logreg = LogisticRegression(solver='liblinear', max_iter=1000, class_weight='balanced')

param_grid = {
    'C': [0.1, 1, 10],
    'penalty': ['l1', 'l2']
}

# Perform Grid Search focusing on Macro-F1 score
grid_search = GridSearchCV(logreg, param_grid, cv=5, scoring='f1_macro')
grid_search.fit(X_train, y_train)

# Final Results Output
print(f"Optimal Parameters: {grid_search.best_params_}")
print(f"Validation Set F1-Score: {grid_search.best_score_:.4f}")

# Model Evaluation on Test Set
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)
print("\n--- Classification Report (Class Weight Balanced) ---")
print(classification_report(y_test, y_pred))

Optimal Parameters: {'C': 1, 'penalty': 'l1'}
Validation Set F1-Score: 0.3674

--- Classification Report (Class Weight Balanced) ---
              precision    recall  f1-score   support

           1       0.18      0.30      0.23        43
           2       0.25      0.23      0.24        91
           3       0.32      0.11      0.16        73
           4       0.79      0.84      0.81       391

    accuracy                           0.62       598
   macro avg       0.39      0.37      0.36       598
weighted avg       0.61      0.62      0.60       598

